In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))


Num GPUs Available:  1



<p dir="rtl">
 התחברות ל־Google Drive :

---



In [17]:
import pandas as pd
import zipfile
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<p dir="rtl">
חילוץ קובץ ZIP ושמירה בתיקייה זמנית ב- colab :



In [18]:
import zipfile

zip_file = '/content/drive/MyDrive/content/CNN_furniture_dataset.zip'

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
  zip_ref.extractall('/content/dataset')


<p dir="rtl">
**הוסר** : הגדרת קונפיגורציה לטעינת תמונות מחוץ למודל :

In [ ]:
# from tensorflow.keras.preprocessing.image import ImageDataGenerator

# train_datagen = ImageDataGenerator(
#     rescale=1./255,
#     shear_range=0.2,
#     zoom_range=0.2,
#     horizontal_flip=True
# )

# test_datagen = ImageDataGenerator(rescale=1./255)



Found 5781 files belonging to 11 classes.
Found 1546 files belonging to 11 classes.


<p dir="rtl">
הגדרת נתיב הקבצים עבור train ו - test מתוך התיקיה הזמנית בתוך colab :

In [4]:
from keras.utils import image_dataset_from_directory

training_set = tf.keras.utils.image_dataset_from_directory(
    '/content/dataset/CNN_furniture_dataset/train',
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

test_set = tf.keras.utils.image_dataset_from_directory(
    '/content/dataset/CNN_furniture_dataset/test',
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

Found 6130 files belonging to 11 classes.
Found 1546 files belonging to 11 classes.


<p dir="rtl">
בניית רשת CNN, כולל הגדרת אוגמנטציה (מחליף את שלב הקונפיגורציה מחוץ למודל):

In [13]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Rescaling, RandomFlip, RandomZoom, GlobalAveragePooling2D, RandomRotation, RandomTranslation # LayerNormalization
from tensorflow.keras.regularizers import l2

cnn = Sequential()

# שלב ה-Augmentation והנרמול
cnn.add(Rescaling(1./255, input_shape=[224, 224, 3]))
cnn.add(RandomFlip("horizontal"))
cnn.add(RandomZoom(0.2))
cnn.add(RandomRotation(0.05)) # סיבוב קל מאוד
cnn.add(RandomTranslation(height_factor=0.1, width_factor=0.1))


# שלב קונבולוציה ראשון
cnn.add(Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

# שלב קונבולוציה שני
cnn.add(Conv2D(filters=64, kernel_size=3, activation='relu'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

# שלב קונבלוציה שלישי
cnn.add(Conv2D(filters=128, kernel_size=3, activation='relu'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

# שלב קונבלוציה רביעי
cnn.add(Conv2D(filters=256, kernel_size=3, activation='relu'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

# שלב קונבלוציה חמישי
cnn.add(Conv2D(filters=512, kernel_size=3, activation='relu'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

# שיטוח התמונה
# cnn.add(Flatten())

# Replacing Flatten with GlobalAveragePooling2D:
# Instead of learning the features precisely as they appear in the training images,
# the GlobalAveragePooling2D function converts all the values ​​of each feature into a single value.
# This allows the model to learn whether a feature is present or absent, without "forcing" it to be identical to the training images.
cnn.add(GlobalAveragePooling2D())

# השכבה הנסתרת המלאה
cnn.add(Dense(units=256, activation='relu'))

# שכבת ה-Dropout החדשה למניעת שינון (Overfitting)
cnn.add(Dropout(0.2))

# 4. שכבת הפלט הסופית ל-11 מחלקות
cnn.add(Dense(units=11, activation='softmax'))



<p dir="rtl">
קומפילציה ואימון המודל:

In [14]:
cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trained_cnn = cnn.fit(
    x=training_set,
    validation_data=test_set,
    epochs=30
)


Epoch 1/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 31s 152ms/step - accuracy: 0.2560 - loss: 2.1330 - val_accuracy: 0.2820 - val_loss: 2.0537
Epoch 2/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 30s 154ms/step - accuracy: 0.3052 - loss: 2.0181 - val_accuracy: 0.3648 - val_loss: 1.8253
Epoch 3/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 44s 168ms/step - accuracy: 0.3990 - loss: 1.8213 - val_accuracy: 0.4217 - val_loss: 1.7459
Epoch 4/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 33s 170ms/step - accuracy: 0.4204 - loss: 1.7581 - val_accuracy: 0.4107 - val_loss: 1.7294
Epoch 5/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 37s 150ms/step - accuracy: 0.4480 - loss: 1.6637 - val_accuracy: 0.4605 - val_loss: 1.6222
Epoch 6/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 29s 150ms/step - accuracy: 0.4938 - loss: 1.5583 - val_accuracy: 0.4573 - val_loss: 1.6905
Epoch 7/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 29s 151ms/step - accuracy: 0.5090 - loss: 1.4907 - val_accuracy: 0.4683 - val_loss: 1.5951
Epoch 8/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 41s 150ms/step - accuracy: 0.5354 - loss: 1

<p dir="rtl">
שמירת המודל:

In [15]:
cnn.save('/content/drive/MyDrive/furniture_cnn_model_3.keras')